In [1]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score

from data_utils import load_data_for_n, get_fold_split, N_LEVELS

MODELS_DIR = Path("../Models/MIL")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
class MILClassifier(nn.Module):
    """
    Multiple Instance Neural Network (MINN), per paper Section 3.4.

    Architecture:
      1. Per-segment: pass each of the N segments through the same 3-layer FFNN
         to produce a 128-dim hidden representation per segment.
      2. MIL max-pooling: for each of the 128 dimensions, take the MAX
         across all N segments in the bag. This is the "witness instance" step
         — the strongest signal in any single segment wins for that dimension.
      3. Final classification: pass the pooled 128-dim bag representation
         through a linear + sigmoid layer to produce ONE probability for
         the whole speaker.
    """
    def __init__(self, input_dim=768):
        super().__init__()
        # Encoder — matches the paper's FFNN spec exactly (32→64→128, ReLU)
        # but STOPS at the last hidden layer, no per-segment output.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),        nn.ReLU(),
            nn.Linear(64, 128),       nn.ReLU(),
        )
        # Final classifier — sees the pooled bag representation, not segments
        self.final = nn.Sequential(
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, bag):
        """
        bag: tensor of shape (N, 768) — the N segment embeddings for one speaker.
        Returns: single probability, shape (1,).
        """
        # Step 1: encode each segment independently -> (N, 128)
        segment_reprs = self.encoder(bag)

        # Step 2: MIL max-pooling across segments (dim=0)
        # For each of the 128 dimensions, take the max value across all N segments.
        # This is the "witness instance" mechanism — one strong segment can
        # drive the bag prediction, even if the others look neutral.
        bag_repr, _ = segment_reprs.max(dim=0)   # -> (128,)

        # Step 3: single classification for the whole bag
        return self.final(bag_repr)              # -> (1,)

In [3]:
def group_into_bags(X, y, speaker):
    """
    Convert flat (num_segments, 768) arrays into speaker-level bags.

    Returns:
        bags   : list of tensors, one per speaker, each shape (N_segments, 768)
        labels : ndarray of shape (num_speakers,) — one label per speaker
    """
    bags, labels = [], []
    for spk in np.unique(speaker):
        mask = (speaker == spk)
        bags.append(torch.tensor(X[mask], dtype=torch.float32))
        labels.append(y[mask][0])   # all segments share the same speaker label
    return bags, np.array(labels)

In [4]:
def train_mil(bags_train, y_train, seed, epochs=30, lr=0.001):
    """
    Train the MIL classifier one bag at a time.
    Note: we train on individual speakers (bags), NOT segments — so
    there are ~92 training examples per fold, not thousands.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MILClassifier(input_dim=bags_train[0].shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    n_speakers = len(bags_train)

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_speakers)   # shuffle speaker order each epoch

        for i in perm:
            bag = bags_train[i]
            label = y_tensor[i]

            optimizer.zero_grad()
            output = model(bag)              # single prob for this speaker
            loss = criterion(output.unsqueeze(0), label.unsqueeze(0))
            loss.backward()
            optimizer.step()

    return model

In [5]:
def evaluate_mil(model, bags_test, y_test):
    """
    Evaluate MIL: one prediction per speaker (bag), no majority vote needed.
    """
    model.eval()
    preds = []

    with torch.no_grad():
        for bag in bags_test:
            output = model(bag).item()       # single probability for this bag
            preds.append(int(output > 0.5))  # threshold 0.5

    preds = np.array(preds)
    acc = (preds == y_test).mean()
    f1 = f1_score(y_test, preds)
    return acc, f1

In [6]:
def run_mil_cv(n, seeds=(0, 1, 2, 3, 4), epochs=30):
    """R=5 protocol matching your FFNN sweep, adapted for MIL."""
    X, y, fold, speaker = load_data_for_n(n)

    accs_per_fold, f1s_per_fold = [], []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(
            X, y, fold, speaker, fold_number
        )

        # Group flat arrays into speaker-level bags for BOTH train and test
        _, _, _, _, speaker_train_full = get_fold_split(X, y, fold, speaker, fold_number)
        # Need speaker info for train side too — refetch
        train_mask = (fold != fold_number)
        speaker_train = speaker[train_mask]

        bags_train, labels_train = group_into_bags(X_train, y_train, speaker_train)
        bags_test, labels_test   = group_into_bags(X_test,  y_test,  speaker_test)

        # R=5 average within each fold
        fold_accs, fold_f1s = [], []
        for seed in seeds:
            model = train_mil(bags_train, labels_train, seed=seed, epochs=epochs)
            acc, f1 = evaluate_mil(model, bags_test, labels_test)
            fold_accs.append(acc); fold_f1s.append(f1)

        accs_per_fold.append(np.mean(fold_accs))
        f1s_per_fold.append(np.mean(fold_f1s))

    return {
        "acc_mean": np.mean(accs_per_fold), "acc_std": np.std(accs_per_fold),
        "f1_mean":  np.mean(f1s_per_fold),  "f1_std":  np.std(f1s_per_fold),
    }

In [7]:
result = run_mil_cv(n=8)
print(f"N=8 MIL: Acc={result['acc_mean']:.4f}±{result['acc_std']:.4f}, "
      f"F1={result['f1_mean']:.4f}±{result['f1_std']:.4f}")

N=8 MIL: Acc=0.8782±0.0531, F1=0.8886±0.0445


In [8]:
results_mil = {}

for n in tqdm(N_LEVELS, desc="N-levels (MIL)"):
    print(f"\n=== N={n} ===")
    results_mil[n] = run_mil_cv(n=n)
    r = results_mil[n]
    print(f"  MIL: Acc={r['acc_mean']:.4f}±{r['acc_std']:.4f}, "
          f"F1={r['f1_mean']:.4f}±{r['f1_std']:.4f}")

# Save
with open(MODELS_DIR / "results_summary.json", "w") as f:
    json.dump({str(n): r for n, r in results_mil.items()}, f, indent=2)

print("\nSaved to Models/MIL/results_summary.json")

N-levels (MIL):   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


N-levels (MIL):  14%|█▍        | 1/7 [00:20<02:01, 20.17s/it]

  MIL: Acc=0.8950±0.0509, F1=0.9089±0.0338

=== N=2 ===


N-levels (MIL):  29%|██▊       | 2/7 [00:41<01:43, 20.62s/it]

...

  MIL: Acc=0.8935±0.0469, F1=0.9038±0.0314

=== N=32 ===


N-levels (MIL):  86%|████████▌ | 6/7 [02:10<00:22, 22.43s/it]

  MIL: Acc=0.8843±0.0750, F1=0.8934±0.0519

=== N=64 ===


N-levels (MIL): 100%|██████████| 7/7 [02:33<00:00, 21.97s/it]

  MIL: Acc=0.8795±0.0430, F1=0.8934±0.0279

Saved to Models/MIL/results_summary.json


In [9]:
### majority class

def run_majority_baseline():
    """
    Compute the majority-class baseline at the speaker (recording) level,
    per fold, using the training fold's class distribution to decide the
    'majority prediction'.

    Uses N=1 loading purely because we only need one row per speaker
    (labels don't change with N, and every N-level has all 116 speakers).
    """
    X, y, fold, speaker = load_data_for_n(1)

    # Reduce to unique speakers (one row per speaker)
    unique_spk = np.unique(speaker)
    spk_labels = np.array([y[speaker == s][0] for s in unique_spk])
    spk_folds  = np.array([fold[speaker == s][0] for s in unique_spk])

    fold_accs, fold_f1s = [], []
    for fold_number in [1, 2, 3, 4, 5]:
        train_mask = (spk_folds != fold_number)
        test_mask  = (spk_folds == fold_number)

        # Predict whichever class is more common in the TRAINING fold
        train_labels = spk_labels[train_mask]
        majority_class = int(np.mean(train_labels) > 0.5)   # 0 or 1

        test_labels = spk_labels[test_mask]
        preds = np.full_like(test_labels, majority_class)

        fold_accs.append((preds == test_labels).mean())
        fold_f1s.append(f1_score(test_labels, preds, zero_division=0))

    print(f"Majority-class baseline (recording-level, 5-fold):")
    print(f"  Accuracy: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
    print(f"  F1:       {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
    return np.mean(fold_accs), np.std(fold_accs), np.mean(fold_f1s), np.std(fold_f1s)

run_majority_baseline()

Majority-class baseline (recording-level, 5-fold):
  Accuracy: 0.4391 ± 0.1186
  F1:       0.5284 ± 0.2668


(0.4391304347826087,
 0.11859288432161613,
 0.5284016636957813,
 0.2668341698731693)

In [10]:
X, y, fold, speaker = load_data_for_n(1)
unique_spk = np.unique(speaker)
spk_labels = np.array([y[speaker == s][0] for s in unique_spk])
spk_folds  = np.array([fold[speaker == s][0] for s in unique_spk])

print(f"Total speakers: {len(unique_spk)}")
print(f"Total: HC={np.sum(spk_labels==0)}, PT={np.sum(spk_labels==1)}\n")

for fn in [1, 2, 3, 4, 5]:
    train_mask = (spk_folds != fn)
    test_mask  = (spk_folds == fn)
    train_hc = np.sum(spk_labels[train_mask] == 0)
    train_pt = np.sum(spk_labels[train_mask] == 1)
    test_hc  = np.sum(spk_labels[test_mask] == 0)
    test_pt  = np.sum(spk_labels[test_mask] == 1)
    train_maj = int(np.mean(spk_labels[train_mask]) > 0.5)
    test_maj  = int(np.mean(spk_labels[test_mask]) > 0.5)
    print(f"Fold {fn}: train HC={train_hc}, PT={train_pt} → majority={train_maj} "
          f"| test HC={test_hc}, PT={test_pt} → true majority={test_maj}")

Total speakers: 116
Total: HC=52, PT=64

Fold 1: train HC=40, PT=52 → majority=1 | test HC=12, PT=12 → true majority=0
Fold 2: train HC=42, PT=51 → majority=1 | test HC=10, PT=13 → true majority=1
Fold 3: train HC=39, PT=54 → majority=1 | test HC=13, PT=10 → true majority=0
Fold 4: train HC=47, PT=46 → majority=0 | test HC=5, PT=18 → true majority=1
Fold 5: train HC=40, PT=53 → majority=1 | test HC=12, PT=11 → true majority=0


In [11]:
def run_always_majority_baseline():
    """
    Simplest baseline: always predict the corpus-wide majority class (PT).
    Ignores fold structure. This matches the '55.2%' number typically
    reported in the literature.
    """
    X, y, fold, speaker = load_data_for_n(1)
    unique_spk = np.unique(speaker)
    spk_labels = np.array([y[speaker == s][0] for s in unique_spk])

    # Always predict PT (label 1)
    preds = np.ones_like(spk_labels)

    acc = (preds == spk_labels).mean()
    f1 = f1_score(spk_labels, preds, zero_division=0)
    print(f"Always-predict-majority (PT) baseline:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1:       {f1:.4f}")
    return acc, f1

run_always_majority_baseline()

Always-predict-majority (PT) baseline:
  Accuracy: 0.5517
  F1:       0.7111


(0.5517241379310345, 0.7111111111111111)

In [12]:
class MILClassifierPooled(nn.Module):
    """
    Identical to MILClassifier, but the pooling operator is configurable.
      pooling="max"  -> selection (original MIL)
      pooling="mean" -> aggregation control
    Encoder, final layer and training are unchanged, so any difference in
    results is attributable to the pooling step alone.
    """
    def __init__(self, input_dim=768, pooling="max"):
        super().__init__()
        self.pooling = pooling
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),        nn.ReLU(),
            nn.Linear(64, 128),       nn.ReLU(),
        )
        self.final = nn.Sequential(nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, bag):
        segment_reprs = self.encoder(bag)              # (N, 128)
        if self.pooling == "max":
            bag_repr, _ = segment_reprs.max(dim=0)
        elif self.pooling == "mean":
            bag_repr = segment_reprs.mean(dim=0)
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")
        return self.final(bag_repr)

In [13]:
def train_mil_pooled(bags_train, y_train, seed, pooling="max", epochs=30, lr=0.001):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MILClassifierPooled(input_dim=bags_train[0].shape[1], pooling=pooling)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

    for epoch in range(epochs):
        model.train()
        for i in torch.randperm(len(bags_train)):
            optimizer.zero_grad()
            loss = criterion(model(bags_train[i]).unsqueeze(0), y_tensor[i].unsqueeze(0))
            loss.backward(); optimizer.step()
    return model


def run_mil_cv_pooled(n, pooling="max", seeds=(0,1,2,3,4), epochs=30):
    X, y, fold, speaker = load_data_for_n(n)
    accs_per_fold, f1s_per_fold = [], []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(
            X, y, fold, speaker, fold_number
        )
        speaker_train = speaker[fold != fold_number]

        bags_train, labels_train = group_into_bags(X_train, y_train, speaker_train)
        bags_test,  labels_test  = group_into_bags(X_test,  y_test,  speaker_test)

        fold_accs, fold_f1s = [], []
        for seed in seeds:
            model = train_mil_pooled(bags_train, labels_train, seed=seed,
                                     pooling=pooling, epochs=epochs)
            acc, f1 = evaluate_mil(model, bags_test, labels_test)
            fold_accs.append(acc); fold_f1s.append(f1)

        accs_per_fold.append(np.mean(fold_accs))
        f1s_per_fold.append(np.mean(fold_f1s))

    return {
        "acc_mean": np.mean(accs_per_fold), "acc_std": np.std(accs_per_fold),
        "f1_mean":  np.mean(f1s_per_fold),  "f1_std":  np.std(f1s_per_fold),
        "fold_f1s": f1s_per_fold, "fold_accs": accs_per_fold,
    }


results_minn = {"max": {}, "mean": {}}
for pooling in ["max", "mean"]:
    print(f"\n===== MINN {pooling.upper()}-POOLING =====")
    for n in tqdm(N_LEVELS, desc=f"MINN-{pooling}", leave=False):
        results_minn[pooling][n] = run_mil_cv_pooled(n=n, pooling=pooling)

print(f"\n{'N':>4} | {'MINN max':>16} | {'MINN mean':>16}")
print("-" * 44)
for n in N_LEVELS:
    a, b = results_minn["max"][n], results_minn["mean"][n]
    print(f"{n:>4} | {a['f1_mean']*100:6.2f}% ± {a['f1_std']*100:4.2f}% "
          f"| {b['f1_mean']*100:6.2f}% ± {b['f1_std']*100:4.2f}%")


===== MINN MAX-POOLING =====



===== MINN MEAN-POOLING =====



   N |         MINN max |        MINN mean
--------------------------------------------
   1 |  90.89% ± 3.38% |  90.89% ± 3.38%
   2 |  90.89% ± 3.11% |  90.75% ± 3.80%
   4 |  91.33% ± 4.03% |  92.05% ± 3.40%
   8 |  88.86% ± 4.45% |  92.85% ± 3.73%
  16 |  90.38% ± 3.14% |  89.73% ± 3.63%
  32 |  89.34% ± 5.19% |  89.26% ± 5.71%
  64 |  89.34% ± 2.79% |  90.69% ± 6.92%


In [14]:
import json
from pathlib import Path

Path("../Models/MIL").mkdir(parents=True, exist_ok=True)
with open("../Models/MIL/results_minn_pooled.json", "w") as f:
    json.dump(
        {pool: {str(n): {k: (float(v) if np.isscalar(v) else [float(x) for x in v])
                         for k, v in r.items()}
                for n, r in res.items()}
         for pool, res in results_minn.items()},
        f, indent=2
    )
print("Saved MINN results with per-fold F1s")

Saved MINN results with per-fold F1s
